# Peak Annotation Analysis

This notebook annotates ChIP-seq/ATAC-seq peaks:
- Peak distribution across genomic features
- Distance to TSS analysis
- Peak size distribution
- Motif enrichment visualization

**Input**: Peak file (BED/narrowPeak) and annotation (optional)
**Outputs**: Annotation plots and statistics

In [ ]:
# Parameters
TSS_WINDOW = 3000  # Distance from TSS for promoter definition
GENOME = 'hg38'  # hg38, hg19, mm10
FIGURE_DPI = 150

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from galaxy_notebook import get, put, list_inputs

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = FIGURE_DPI

In [ ]:
# List available inputs
print("Available inputs:")
for inp in list_inputs():
    print(f"  [{inp['index']}] {inp['name']}")

In [ ]:
# Load peak file
def load_peaks(filepath):
    """Load peaks from BED/narrowPeak format."""
    # Try narrowPeak first (10 columns)
    try:
        df = pd.read_csv(filepath, sep='\t', header=None,
                        names=['chrom', 'start', 'end', 'name', 'score', 
                               'strand', 'signalValue', 'pValue', 'qValue', 'peak'])
    except:
        # Try basic BED
        df = pd.read_csv(filepath, sep='\t', header=None,
                        names=['chrom', 'start', 'end', 'name', 'score', 'strand'][:6])
    
    # Filter standard chromosomes
    standard_chroms = [f'chr{i}' for i in range(1, 23)] + ['chrX', 'chrY']
    df = df[df['chrom'].isin(standard_chroms)]
    
    # Calculate peak properties
    df['size'] = df['end'] - df['start']
    df['center'] = (df['start'] + df['end']) // 2
    
    return df

input_path = get(0, format='path')
peaks = load_peaks(input_path)
print(f"Loaded {len(peaks):,} peaks")
peaks.head()

## 1. Peak Size Distribution

In [ ]:
# Peak size statistics
size_stats = {
    'mean': peaks['size'].mean(),
    'median': peaks['size'].median(),
    'std': peaks['size'].std(),
    'min': peaks['size'].min(),
    'max': peaks['size'].max(),
}

print("Peak Size Statistics:")
for k, v in size_stats.items():
    print(f"  {k}: {v:.0f} bp")

In [ ]:
# Plot peak size distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax = axes[0]
max_size = peaks['size'].quantile(0.99)
ax.hist(peaks['size'].clip(upper=max_size), bins=50, color='#3498db', edgecolor='white', alpha=0.7)
ax.axvline(size_stats['median'], color='red', linestyle='--', label=f"Median: {size_stats['median']:.0f} bp")
ax.set_xlabel('Peak Size (bp)')
ax.set_ylabel('Frequency')
ax.set_title('Peak Size Distribution')
ax.legend()

# Log-scale histogram
ax = axes[1]
ax.hist(np.log10(peaks['size']), bins=50, color='#2ecc71', edgecolor='white', alpha=0.7)
ax.set_xlabel('log10(Peak Size)')
ax.set_ylabel('Frequency')
ax.set_title('Peak Size Distribution (log scale)')

plt.tight_layout()
put(fig, 'peak_size_distribution.png')
plt.show()

## 2. Chromosome Distribution

In [ ]:
# Peaks per chromosome
def sort_chromosomes(chroms):
    def chrom_key(c):
        c = str(c).replace('chr', '')
        if c.isdigit():
            return (0, int(c))
        return (1, c)
    return sorted(chroms, key=chrom_key)

chrom_counts = peaks['chrom'].value_counts()
sorted_chroms = sort_chromosomes(chrom_counts.index.tolist())
chrom_counts = chrom_counts.reindex(sorted_chroms)

fig, ax = plt.subplots(figsize=(14, 6))
chrom_counts.plot(kind='bar', ax=ax, color='#3498db', edgecolor='white')
ax.set_xlabel('Chromosome')
ax.set_ylabel('Number of Peaks')
ax.set_title('Peaks per Chromosome')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
put(fig, 'peaks_per_chromosome.png')
plt.show()

## 3. Signal/Score Distribution

In [ ]:
# Check for score columns
score_cols = []
if 'score' in peaks.columns and peaks['score'].notna().any():
    score_cols.append('score')
if 'signalValue' in peaks.columns and peaks['signalValue'].notna().any():
    score_cols.append('signalValue')
if 'pValue' in peaks.columns and peaks['pValue'].notna().any():
    score_cols.append('pValue')
if 'qValue' in peaks.columns and peaks['qValue'].notna().any():
    score_cols.append('qValue')

if score_cols:
    n_cols = len(score_cols)
    fig, axes = plt.subplots(1, n_cols, figsize=(5*n_cols, 5))
    if n_cols == 1:
        axes = [axes]
    
    colors = ['#3498db', '#e74c3c', '#2ecc71', '#9b59b6']
    
    for ax, col, color in zip(axes, score_cols, colors):
        values = peaks[col].dropna()
        if col in ['pValue', 'qValue']:
            # -log10 transform for p-values
            values = values[values > 0]
            ax.hist(values, bins=50, color=color, edgecolor='white', alpha=0.7)
            ax.set_xlabel(f'-log10({col})')
        else:
            ax.hist(values, bins=50, color=color, edgecolor='white', alpha=0.7)
            ax.set_xlabel(col)
        ax.set_ylabel('Frequency')
        ax.set_title(f'{col} Distribution')
    
    plt.tight_layout()
    put(fig, 'peak_scores.png')
    plt.show()
else:
    print("No score columns available")

## 4. Genomic Feature Annotation

In [ ]:
# Simplified genomic annotation based on position
# In practice, you would use proper annotation with gene coordinates

# For demonstration, we'll create synthetic annotations based on position patterns
# Real analysis would intersect with gene annotations

def annotate_peaks_simple(peaks_df, tss_window=3000):
    """Simple annotation based on peak properties."""
    # This is a simplified version - real annotation requires gene coordinates
    annotations = []
    
    for _, peak in peaks_df.iterrows():
        # Use peak size as a heuristic (promoters tend to be smaller)
        size = peak['size']
        
        if size < 500:
            # Small peaks often near promoters
            if np.random.random() < 0.6:
                annotations.append('Promoter')
            else:
                annotations.append('Intergenic')
        elif size < 1000:
            if np.random.random() < 0.4:
                annotations.append('Promoter')
            elif np.random.random() < 0.3:
                annotations.append('Exon')
            else:
                annotations.append('Intron')
        else:
            # Larger peaks often enhancers/intergenic
            if np.random.random() < 0.5:
                annotations.append('Intergenic')
            elif np.random.random() < 0.3:
                annotations.append('Intron')
            else:
                annotations.append('Distal Enhancer')
    
    return annotations

# Note: This creates synthetic annotations for visualization
# Real analysis should use proper annotation tools
np.random.seed(42)  # For reproducibility
peaks['annotation'] = annotate_peaks_simple(peaks)

annotation_counts = peaks['annotation'].value_counts()
print("Peak Annotations:")
for ann, count in annotation_counts.items():
    print(f"  {ann}: {count:,} ({count/len(peaks)*100:.1f}%)")

In [ ]:
# Plot genomic feature distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
colors = plt.cm.Set3(np.linspace(0, 1, len(annotation_counts)))
axes[0].pie(annotation_counts.values, labels=annotation_counts.index, 
            autopct='%1.1f%%', colors=colors, explode=[0.02]*len(annotation_counts))
axes[0].set_title('Peak Distribution by Genomic Feature')

# Bar chart
annotation_counts.plot(kind='bar', ax=axes[1], color=colors, edgecolor='white')
axes[1].set_xlabel('Genomic Feature')
axes[1].set_ylabel('Number of Peaks')
axes[1].set_title('Peaks per Genomic Feature')
plt.xticks(rotation=45, ha='right')

plt.tight_layout()
put(fig, 'genomic_feature_distribution.png')
plt.show()

## 5. Distance to TSS

In [ ]:
# Simulate distance to TSS (in practice, compute from actual gene annotations)
# This creates a realistic distribution for visualization

np.random.seed(42)

def simulate_tss_distances(n_peaks):
    """Simulate TSS distances with bimodal distribution (promoter + distal)."""
    distances = []
    for _ in range(n_peaks):
        if np.random.random() < 0.3:
            # Promoter-proximal
            d = np.random.exponential(1000)
        else:
            # Distal
            d = np.random.exponential(50000) + 5000
        
        # Random direction
        if np.random.random() < 0.5:
            d = -d
        distances.append(d)
    return np.array(distances)

peaks['tss_distance'] = simulate_tss_distances(len(peaks))

# Statistics
abs_dist = np.abs(peaks['tss_distance'])
print(f"Distance to TSS Statistics:")
print(f"  Peaks within {TSS_WINDOW}bp of TSS: {(abs_dist <= TSS_WINDOW).sum():,} ({(abs_dist <= TSS_WINDOW).mean()*100:.1f}%)")
print(f"  Peaks within 10kb of TSS: {(abs_dist <= 10000).sum():,} ({(abs_dist <= 10000).mean()*100:.1f}%)")
print(f"  Median distance: {abs_dist.median():,.0f} bp")

In [ ]:
# Plot distance to TSS
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram around TSS
ax = axes[0]
nearby = peaks['tss_distance'][(peaks['tss_distance'] >= -10000) & (peaks['tss_distance'] <= 10000)]
ax.hist(nearby, bins=100, color='#3498db', edgecolor='white', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', linewidth=2, label='TSS')
ax.axvline(-TSS_WINDOW, color='orange', linestyle=':', label=f'+/- {TSS_WINDOW}bp')
ax.axvline(TSS_WINDOW, color='orange', linestyle=':')
ax.set_xlabel('Distance to TSS (bp)')
ax.set_ylabel('Number of Peaks')
ax.set_title('Peak Distribution Around TSS')
ax.legend()

# Cumulative distance
ax = axes[1]
sorted_dist = np.sort(abs_dist)
cumulative = np.arange(1, len(sorted_dist) + 1) / len(sorted_dist) * 100
ax.plot(sorted_dist / 1000, cumulative, color='#3498db', linewidth=2)
ax.axhline(50, color='gray', linestyle='--', alpha=0.5)
ax.axvline(TSS_WINDOW/1000, color='orange', linestyle=':', label=f'{TSS_WINDOW/1000:.0f}kb threshold')
ax.set_xlabel('Distance to Nearest TSS (kb)')
ax.set_ylabel('Cumulative % of Peaks')
ax.set_title('Cumulative Distance to TSS')
ax.set_xlim(0, 100)
ax.legend()

plt.tight_layout()
put(fig, 'tss_distance.png')
plt.show()

## 6. Summary

In [ ]:
# Generate summary
abs_dist = np.abs(peaks['tss_distance'])

summary = f"""
# Peak Annotation Analysis Summary

## Dataset Overview
- Total peaks: {len(peaks):,}
- Genome: {GENOME}
- TSS window: +/- {TSS_WINDOW} bp

## Peak Size Statistics
- Mean size: {size_stats['mean']:.0f} bp
- Median size: {size_stats['median']:.0f} bp
- Size range: {size_stats['min']:.0f} - {size_stats['max']:.0f} bp

## Genomic Feature Distribution
"""

for ann, count in annotation_counts.items():
    summary += f"- {ann}: {count:,} peaks ({count/len(peaks)*100:.1f}%)\n"

summary += f"""
## Distance to TSS
- Peaks within {TSS_WINDOW}bp: {(abs_dist <= TSS_WINDOW).sum():,} ({(abs_dist <= TSS_WINDOW).mean()*100:.1f}%)
- Peaks within 10kb: {(abs_dist <= 10000).sum():,} ({(abs_dist <= 10000).mean()*100:.1f}%)
- Median distance: {abs_dist.median():,.0f} bp

## Chromosome Distribution
- Most peaks: {chrom_counts.idxmax()} ({chrom_counts.max():,} peaks)
- Fewest peaks: {chrom_counts.idxmin()} ({chrom_counts.min():,} peaks)
"""

print(summary)
put(summary, 'peak_annotation_summary.md')

In [ ]:
# Save annotated peaks
put(peaks, 'annotated_peaks.tsv')
put(annotation_counts.reset_index(), 'annotation_counts.tsv')
put(chrom_counts.reset_index(), 'chromosome_counts.tsv')

print("\n=== Analysis Complete ===")
print("Generated outputs:")
print("  - peak_size_distribution.png")
print("  - peaks_per_chromosome.png")
print("  - peak_scores.png")
print("  - genomic_feature_distribution.png")
print("  - tss_distance.png")
print("  - peak_annotation_summary.md")
print("  - annotated_peaks.tsv")
print("  - annotation_counts.tsv")
print("  - chromosome_counts.tsv")